In [1]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="distilbert_cls_cosine_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [2]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [3]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)



---[ TableVault Record ]---
device: mps
---[ TableVault Record ]---



In [4]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print(model_name)
print("hidden_size:", model.config.hidden_size)



---[ TableVault Record ]---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased
hidden_size: 768
---[ TableVault Record ]---



In [5]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())



---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
---[ TableVault Record ]---



In [6]:
def pool_first_token(hidden_state):
    pooled = hidden_state[:, 0, :]
    pooled = torch.nn.functional.normalize(pooled, p=2, dim=-1)
    return pooled

@torch.no_grad()
def encode_sentences(texts, batch_size=128, max_length=128):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        outputs = model(**enc)
        pooled = pool_first_token(outputs.last_hidden_state)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()



---[ TableVault Record ]---
---[ TableVault Record ]---



In [7]:
emb1 = encode_sentences(sent1)
emb2 = encode_sentences(sent2)

scores = np.sum(emb1 * emb2, axis=1)
threshold = 0.85
y_pred = (scores >= threshold).astype(int)

print("emb1 shape:", emb1.shape)
print("emb2 shape:", emb2.shape)
print("score range:", float(scores.min()), float(scores.max()))
print("done")



---[ TableVault Record ]---


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

emb1 shape: (408, 768)
emb2 shape: (408, 768)
score range: 0.8139357566833496 0.9993458986282349
done
---[ TableVault Record ]---



In [8]:

vault.create_record_list("cosine_mrpc_threshold_score", column_names=["prediction", "score"])

for i in range(len(y_pred)):
    vault.append_record("cosine_mrpc_threshold_score", 
                        {
                            "prediction": int(y_pred[i]),
                            "score": float(scores[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "Per-example prediction dataset for the GLUE MRPC validation split. Each record corresponds to one sentence pair from glue_mrpc_validation and stores the output of a simple paraphrase classifier built from DistilBERT sentence embeddings. The dataset has two fields: prediction, a binary label where 1 means paraphrase and 0 means not paraphrase, produced by thresholding the cosine similarity at 0.85; and score, the raw cosine similarity between the L2-normalized first-token embeddings of sentence1 and sentence2 from distilbert-base-uncased. In this workflow, this dataset is the main record of model inference results, used for error analysis, inspection of individual examples, and as input to the downstream summary dataset distilbert_cls_cosine_mrpc_summary."
embedding = get_embeddings(description)
vault.create_description("cosine_mrpc_threshold_score", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "model predictions", "prediction_type": "binary classification with cosine similarity score", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "distilbert-base-uncased", "embedding_representation": "first-token CLS-like embedding", "similarity_metric": "cosine similarity", "decision_rule": "threshold >= 0.85", "input_fields": "sentence1,sentence2", "output_fields": "prediction,score", "label_space": "0=not_paraphrase,1=paraphrase", "framework": "transformers+pytorch", "upstream_item": "glue_mrpc_validation", "process": "distilbert_cls_cosine_mrpc"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("cosine_mrpc_threshold_score", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [9]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))



---[ TableVault Record ]---
{'accuracy': 0.6862745098039216, 'f1': 0.8134110787172012}
                precision    recall  f1-score   support

not_paraphrase       1.00      0.01      0.02       129
    paraphrase       0.69      1.00      0.81       279

      accuracy                           0.69       408
     macro avg       0.84      0.50      0.41       408
  weighted avg       0.78      0.69      0.56       408

---[ TableVault Record ]---



In [10]:
for i in range(5):
    print("=" * 80)
    print("idx:", i)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("cosine_score:", float(scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("cosine_score:", float(scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))



---[ TableVault Record ]---
idx: 0
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
cosine_score: 0.937690019607544
true: 1 pred: 1
idx: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
cosine_score: 0.9651895761489868
true: 0 pred: 1
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
cosine_score: 0.9854921102523804
true: 0 pred: 1
idx: 3
sentence1: The AFL-CIO is waiting until October to decide if it will endo

In [11]:
vault.create_record_list("distilbert_cls_cosine_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("distilbert_cls_cosine_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "cosine_mrpc_threshold_score": [0, len(ds)]
                    })

summary

description = "Summary dataset for the DistilBERT CLS cosine-similarity evaluation on the GLUE MRPC validation split. It contains aggregate evaluation results computed by comparing thresholded cosine-similarity predictions from sentence-pair embeddings against the ground-truth MRPC labels. The record list has three fields: accuracy (float), f1 (float), and classification_report (string with per-class precision/recall/F1 and support). In this workflow, it serves as the experiment-level evaluation artifact: a compact summary of model performance derived from the glue_mrpc_validation inputs and the per-example predictions stored in cosine_mrpc_threshold_score."
embedding = get_embeddings(description)
vault.create_description("distilbert_cls_cosine_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "evaluation summary", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "distilbert-base-uncased", "representation": "cls first-token embedding", "similarity_metric": "cosine similarity", "decision_rule": "threshold", "threshold": "0.85", "framework": "transformers", "metrics": "accuracy,f1,classification_report", "input": "sentence pairs", "labels": "binary paraphrase/non-paraphrase", "upstream_record_list": "cosine_mrpc_threshold_score", "process": "distilbert_cls_cosine_mrpc"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_cls_cosine_mrpc_summary", cat, embedding, prop)




---[ TableVault Record ]---
---[ TableVault Record ]---



In [12]:
description = "This notebook evaluates a simple paraphrase detection workflow on the GLUE MRPC validation set using DistilBERT sentence representations and cosine similarity. It loads sentence pairs and labels from TableVault, encodes each sentence with distilbert-base-uncased, uses the first-token ([CLS]-style) hidden state as a normalized embedding, computes cosine similarity between the two sentence embeddings, and applies a fixed threshold of 0.85 to predict whether the pair is a paraphrase. The notebook then measures performance with accuracy, F1, and a classification report, prints example predictions and errors for inspection, and stores per-example scores, summary metrics, and semantic descriptions back into TableVault for experiment tracking and retrieval." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("distilbert_cls_cosine_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "method": "sentence pair classification via cosine similarity", "model": "distilbert-base-uncased", "representation": "CLS first-token embedding", "similarity_metric": "cosine similarity", "decision_rule": "fixed threshold 0.85", "dataset": "glue/mrpc validation", "split": "validation", "framework": "PyTorch, Hugging Face Transformers", "evaluation": "accuracy, f1-score, classification report", "tracking": "TableVault with ArangoDB", "metadata_embedding_model": "text-embedding-3-large"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_cls_cosine_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

